In [ ]:
import os
import loompy as lp
import numpy as np
import scanpy as sc
import glob
import pickle
import pandas as pd
import seaborn as sns

from pathlib import Path
from dask.diagnostics import ProgressBar
from arboreto.utils import load_tf_names
from arboreto.algo import grnboost2
from ctxcore.rnkdb import FeatherRankingDatabase as RankingDatabase
from pyscenic.utils import modules_from_adjacencies, load_motifs
from pyscenic.prune import prune2df, df2regulons

#### 1. Convert CSV expression matrix to Loom format

In [ ]:
input_dir = "/home/fuyq/GRN/single_cell/scenic/PBMC10k/count/"
output_dir = "/home/fuyq/GRN/single_cell/scenic/PBMC10k/loom"

csv_files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]

for csv_file in csv_files:
    file_path = os.path.join(input_dir, csv_file)
    x = sc.read_csv(file_path)

    row_attrs = {"Gene": np.array(x.var_names)}
    col_attrs = {"CellID": np.array(x.obs_names)}

    output_file = os.path.join(output_dir, f"{csv_file.split('.')[0]}.loom")

    lp.create(output_file, x.X.transpose(), row_attrs, col_attrs)

    print(f"Created loom file: {output_file}")

#### 2. Run GRN inference using GRNBoost2

In [ ]:
files = glob.glob("/home/fuyq/GRN/single_cell/scenic/PBMC10k/loom/*.loom")
celltypes = [Path(f).stem for f in files]
TF_FILE = "/home/fuyq/GRN/GTEx/tf_hg_new.txt"

In [ ]:
for celltype in celltypes:
    print(f"Running {celltype}...")
 
    cmd = f"""
    pyscenic grn \
        --num_workers 32 \
        --output /home/fuyq/GRN/single_cell/scenic/PBMC10k/adj/adj_{celltype}.tsv \
        --method grnboost2 \
        /home/fuyq/GRN/single_cell/scenic/PBMC10k/loom/{celltype}.loom \
        {TF_FILE}
    """
    !{cmd}

#### 3. Run cisTarget

In [ ]:
FEATHER_folder = "/home/fuyq/cisTarget_databases/homo_sapiens/hg38/"
FEATHER_glob = os.path.join(FEATHER_folder, "hg38__refseq-r80__*.mc9nr.genes_vs_motifs.rankings.feather")
MOTIF_ANNOTATIONS = os.path.join("/home/fuyq/tbl", "motifs-v9-nr.hgnc-m0.001-o0.0.tbl")

In [ ]:
for celltype in celltypes:
    print(f"Running {celltype}...")
 
    cmd = f"""
    pyscenic ctx \
        /home/fuyq/GRN/single_cell/scenic/PBMC10k/adj/adj_{celltype}.tsv \
        {FEATHER_glob} \
        --annotations_fname {MOTIF_ANNOTATIONS} \
        --expression_mtx_fname /home/fuyq/GRN/single_cell/scenic/PBMC10k/loom/{celltype}.loom \
        --output /home/fuyq/GRN/single_cell/scenic/PBMC10k/ctx/reg_{celltype}.gmt \
        --mask_dropouts \
        --num_workers 32
    """
    !{cmd}